# Artificial Data

We generate random numbers ensuring that variables A and B exhibit correlation, while variable C remains uncorrelated. The proposed sampling function is capable of producing data that adheres to the same underlying patterns observed in the original dataset.

In [ ]:
import pandas as pd
import numpy as np

size = 1000
A = np.random.randint(100, size=size)
B = A // 10
C = np.random.randint(100, size=size)
df = pd.DataFrame(np.vstack([A,B,C]).T, columns=['A','B','C'])
df

In [ ]:
from dataframe_sampler import ConcreteDataFrameSampler
sampler = ConcreteDataFrameSampler(n_bins=10,n_neighbours=3)
sampler.fit(df)
generated_df = sampler.sample(n_samples=len(df))
generated_df

---

# Real Data

We consider real data from https://www.kaggle.com/datasets/nelgiriyewithana/billionaires-statistics-dataset


In [ ]:
df = pd.read_csv('data/BillionairesStatisticsDataset.csv')
df = df.fillna(0)
df

We select desired columns and add numerical columns (e.g. numerical encoding for country)

In [ ]:
# Importing the LabelEncoder from sklearn's preprocessing module
from sklearn.preprocessing import LabelEncoder

# Create a copy of the original DataFrame with only the specified columns
df = df[['finalWorth', 'category', 'personName', 'age', 'country', 'city']].copy()

# Ensure that all values in the 'country' column are treated as strings
df['country'] = df['country'].astype(str)

# Split the 'personName' column by spaces and keep only the first part (first name)
df.loc[:, 'personName'] = df['personName'].str.split().str[0]

# Encode the 'country' column with integer values, creating a new 'country_id' column
df.loc[:, 'country_id'] = LabelEncoder().fit_transform(df['country'])

# Display the DataFrame to confirm changes
df

In [ ]:
# Importing the ConcreteDataFrameSampler from the dataframe_sampler module
from dataframe_sampler import ConcreteDataFrameSampler

# Defining the columns to be sampled from the DataFrame
sampled_columns = ['finalWorth', 'category', 'personName', 'city', 'country']

# Determining the number of bins for sampling based on the 'country_id' column
n_bins = df['country_id'].max() * 2

# Initializing the ConcreteDataFrameSampler with the specified number of bins, 
# number of neighbors, sampled columns, and vectorizing columns dictionary
sampler = ConcreteDataFrameSampler(n_bins=n_bins, n_neighbours=5, sampled_columns=sampled_columns)

# Fitting the sampler to the DataFrame
sampler.fit(df)

# Generating a sample of 10 rows from the fitted DataFrame
generated_df = sampler.sample(n_samples=1000)

# Sorting the generated DataFrame by 'finalWorth' column in descending order
generated_df.sort_values(by=['finalWorth'], ascending=False)

---

# Real data

We consider real data from https://www.kaggle.com/datasets/zhaoyingzhu/heartcsv

We aim to demonstrate that the data generated using the proposed sampling function is comparable in quality to the original data for training a classifier. This can be evidenced by evaluating the classifier's predictive performance on an authentic test set.

In [ ]:
df = pd.read_csv('data/heart.csv')
df

In [ ]:
# Extracting feature matrix X and target vector y from the DataFrame
# X contains all columns except the last one
# y contains only the last column
X = df.values[:, :-1]
y = df.values[:, -1]

# Importing the train_test_split function from sklearn's model_selection module
from sklearn.model_selection import train_test_split

# Splitting the dataset into training and testing sets
# 70% of the data is used for training and 30% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7)

# Importing the RandomForestClassifier from sklearn's ensemble module
from sklearn.ensemble import RandomForestClassifier

# Initializing the RandomForestClassifier with 100 trees
# Fitting the classifier to the training data and making predictions on the test data
preds = RandomForestClassifier(n_estimators=300).fit(X_train, y_train).predict(X_test)

# Importing the accuracy_score function from sklearn's metrics module
from sklearn.metrics import accuracy_score

# Calculating the accuracy of the predictions
acc = accuracy_score(y_test, preds)

# Printing the accuracy score with 3 decimal places
print('Acc: %.3f' % acc)

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from dataframe_sampler import ConcreteDataFrameSampler

# Combining the training features and target into a single DataFrame
# np.hstack is used to horizontally stack the X_train array and the reshaped y_train array
df_train = pd.DataFrame(np.hstack([X_train, y_train.reshape(-1, 1)]), columns=df.columns)

# Initializing the ConcreteDataFrameSampler with specified number of bins and neighbors
sampler = ConcreteDataFrameSampler(n_bins=10, n_neighbours=3)

# Fitting the sampler to the training DataFrame
sampler.fit(df_train)

# Generating a sample of the same length as the training DataFrame
generated_df = sampler.sample(n_samples=len(df_train))

# Extracting features and target from the generated sample DataFrame
generated_X = generated_df.values[:, :-1]
generated_y = generated_df.values[:, -1]

# Training a RandomForestClassifier on the generated data
# and making predictions on the original test set
preds = RandomForestClassifier(n_estimators=300).fit(generated_X, generated_y).predict(X_test)

# Calculating the accuracy of the predictions
acc = accuracy_score(y_test, preds)

# Printing the accuracy score with 3 decimal places
print('Acc: %.3f' % acc)

---

---
# OpenAI-assisted Auto Configuration

This optional mode asks an OpenAI model to inspect a compact dataframe profile and recommend `sampled_columns`, an `embedding_method`, and a `knn_backend`. It requires `pip install .[llm]` and `OPENAI_API_KEY` in the environment.


In [ ]:
import os
import pandas as pd
from dataframe_sampler import ConcreteDataFrameSampler, suggest_sampler_config_with_openai

demo_df = pd.DataFrame({
    'personName': ['Ann Lee', 'Bob Kay', 'Cam Fox', 'Dan Yu', 'Eve Oz', 'Flo Ray'],
    'age': [24, 31, 45, 52, 29, 61],
    'country': ['PT', 'PT', 'UK', 'UK', 'FR', 'FR'],
    'country_id': [1, 1, 2, 2, 3, 3],
    'income': [30000, 42000, 65000, 80000, 51000, 90000],
})

if 'OPENAI_API_KEY' not in os.environ:
    print('Set OPENAI_API_KEY and install the llm extra to run this auto-configuration example.')
else:
    config = suggest_sampler_config_with_openai(demo_df, model='gpt-4o-mini')
    print(config)

    sampler = ConcreteDataFrameSampler(
        n_bins=4,
        n_neighbours=2,
        sampled_columns=config['sampled_columns'],
        embedding_method=config['embedding_method'],
        knn_backend='sklearn',
        random_state=42,
    )
    sampler.fit(demo_df)
    generated_df = sampler.sample(n_samples=len(demo_df))
    generated_df
